# DAY 6 - TASK 1
# Historical Value at Risk (VaR) and Conditional VaR (CVaR)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
nav_data = pd.read_csv("../data/processed/clean_nav_history.csv")
scheme_data = pd.read_csv("../data/processed/clean_scheme_performance.csv")
transactions = pd.read_csv("../data/processed/clean_investor_transactions.csv")
holdings_data = pd.read_csv("../data/raw/09_portfolio_holdings.csv")

In [ ]:
nav_data["date"] = pd.to_datetime(nav_data["date"])
nav_data = nav_data.sort_values(by=["amfi_code", "date"])
nav_data["daily_return"] = (nav_data.groupby("amfi_code")["nav"].pct_change())
returns_data = nav_data.dropna(subset=["daily_return"]).copy()
returns_data.head()

In [ ]:
risk_records = []
for fund_code, fund_frame in returns_data.groupby("amfi_code"):
    fund_returns = fund_frame["daily_return"]
    var_95 = np.percentile(fund_returns,5)
    cvar_95 = fund_returns[fund_returns <= var_95].mean()
    risk_records.append({"amfi_code": fund_code,"var_95": var_95,"cvar_95": cvar_95,
        "num_observations": len(fund_returns)}
        )

In [ ]:
var_cvar_report = pd.DataFrame(risk_records)
var_cvar_report.head()

In [ ]:
scheme_data = scheme_data[["amfi_code", "scheme_name"]]
var_cvar_report = (var_cvar_report.merge(
                    scheme_data,
                    on="amfi_code",
                    how="left"
                    )
)

In [ ]:
var_cvar_report = var_cvar_report[[
        "amfi_code",
        "scheme_name",
        "var_95",
        "cvar_95",
        "num_observations"
    ]
]
var_cvar_report.head()

In [ ]:
highest_risk_funds = (var_cvar_report.sort_values(by="var_95"))
highest_risk_funds.head(10)

In [ ]:
var_cvar_report.to_csv("../reports/var_cvar_report.csv",index=False)
print("var_cvar_report.csv saved successfully.")

# DAY 6 - TASK 2
# Rolling 90-Day Sharpe Ratio Analysis

In [ ]:
nav_data["date"] = pd.to_datetime(nav_data["date"])
nav_data = nav_data.sort_values(by=["amfi_code", "date"])
nav_data.head()

In [ ]:
nav_data["daily_return"] = (nav_data.groupby("amfi_code")["nav"].pct_change())
returns_data = nav_data.dropna(subset=["daily_return"]).copy()
returns_data.head()

In [ ]:
scheme_data = pd.read_csv("../data/processed/clean_scheme_performance.csv")
top_funds = (
    scheme_data
    .sort_values(
        by="aum_crore",
        ascending=False
    ).head(5))
selected_funds = (top_funds["amfi_code"].tolist())
top_funds[["amfi_code", "scheme_name", "aum_crore"]]


In [ ]:
rolling_sharpe_frames = []
for fund_code in selected_funds:
    fund_data = (returns_data[returns_data["amfi_code"] == fund_code].copy())
    rolling_mean = (fund_data["daily_return"].rolling(window=90).mean())

    rolling_std = (fund_data["daily_return"].rolling(window=90).std())
    fund_data["rolling_sharpe"] = (rolling_mean /rolling_std) * np.sqrt(252)
    rolling_sharpe_frames.append(fund_data)

In [ ]:
rolling_sharpe_df = pd.concat(rolling_sharpe_frames,ignore_index=True)
rolling_sharpe_df.head()

In [ ]:
fund_lookup = scheme_data[["amfi_code", "scheme_name"]]
rolling_sharpe_df = (rolling_sharpe_df.merge(
        fund_lookup,
        on="amfi_code",
        how="left"
    )
)
rolling_sharpe_df.head()

In [ ]:
plt.figure(figsize=(14, 7))
for fund_name, fund_frame in (rolling_sharpe_df.groupby("scheme_name")):
    plt.plot(
        fund_frame["date"],
        fund_frame["rolling_sharpe"],
        linewidth=2,
        label=fund_name
    )
plt.title("Rolling 90-Day Sharpe Ratio of Top 5 Funds")
plt.xlabel("Date")
plt.ylabel("Rolling Sharpe Ratio")
plt.legend(bbox_to_anchor=(1.05, 1),loc="upper left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../reports/rolling_sharpe_chart.png",dpi=300,bbox_inches="tight")
plt.show()
print("rolling_sharpe_chart.png saved successfully.")

# DAY 6 - TASK 3
# Investor Cohort Analysis

In [ ]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])
transactions.head()

In [ ]:
first_activity = (transactions.groupby("investor_id")["transaction_date"].min().reset_index())
first_activity.rename(columns={"transaction_date": "first_transaction_date"},inplace=True)
first_activity.head()

In [ ]:
first_activity["cohort_year"] = (first_activity["first_transaction_date"].dt.year)
first_activity.head()

In [ ]:
transactions = (transactions.merge(
        first_activity[["investor_id", "cohort_year"]],on="investor_id",how="left"
    )
)
transactions.head()

In [ ]:
average_investment = (
    transactions
    .groupby("cohort_year")["amount_inr"]
    .mean()
    .reset_index()
)
average_investment.rename(columns={"amount_inr": "avg_investment_amount"},inplace=True)
average_investment

In [ ]:
total_investment = (transactions.groupby("cohort_year")["amount_inr"].sum().reset_index())
total_investment.rename(columns={"amount_inr": "total_invested_amount"},inplace=True)
total_investment

In [ ]:
fund_preference = (transactions.groupby(["cohort_year", "amfi_code"]).size()
    .reset_index(name="transaction_count")
)

In [ ]:
scheme_lookup = scheme_data[["amfi_code", "scheme_name"]]
top_fund_data = (fund_preference
    .sort_values(by=["cohort_year","transaction_count"],ascending=[True, False])
    .groupby("cohort_year")
    .first()
    .reset_index()
)
top_fund_data = (top_fund_data.merge(scheme_lookup,on="amfi_code",how="left"))
top_fund_data.head()

In [ ]:
cohort_analysis = (average_investment
    .merge(total_investment,on="cohort_year")
    .merge(top_fund_data[["cohort_year","scheme_name"]],on="cohort_year"))
cohort_analysis.rename(columns={"scheme_name": "top_fund_preference"},inplace=True)
cohort_analysis

In [ ]:
cohort_analysis.to_csv("../reports/cohort_analysis.csv",index=False)
print("cohort_analysis.csv saved successfully.")

# DAY 6 - TASK 4
# SIP Continuity Analysis

In [ ]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])
sip_transactions = (transactions[transactions["transaction_type"].str.upper().eq("SIP")].copy())
sip_transactions.head()

In [ ]:
sip_transactions = sip_transactions.sort_values(by=["investor_id","transaction_date"])
sip_transactions.head()

In [ ]:
sip_transactions["days_between_sips"] = (sip_transactions
    .groupby("investor_id")["transaction_date"]
    .diff()
    .dt.days
)
sip_transactions.head()

In [ ]:
sip_counts = (sip_transactions.groupby("investor_id")
    .size()
    .reset_index(name="sip_transaction_count"))
sip_counts.head()

In [ ]:
eligible_investors = (sip_counts[sip_counts["sip_transaction_count"] >= 6])
eligible_investors.head()

In [ ]:
average_gap_data = (sip_transactions.groupby("investor_id")["days_between_sips"]
    .mean()
    .reset_index()
)
average_gap_data.rename(columns={"days_between_sips": "average_gap_days"},inplace=True)
average_gap_data.head()

In [ ]:
sip_continuity = (eligible_investors.merge(average_gap_data,on="investor_id",how="left"))
sip_continuity.head()

In [ ]:
sip_continuity["sip_status"] = (sip_continuity["average_gap_days"]
    .apply(
        lambda gap:
        "At-Risk"
        if gap > 35
        else "Active")
)
sip_continuity["sip_status"].value_counts()

In [ ]:
sip_continuity.to_csv("../reports/sip_continuity.csv",index=False)
print("sip_continuity.csv saved successfully.")

# DAY 6 - TASK 6
# Sector Concentration Analysis (HHI)

In [ ]:
sector_summary = (holdings_data.groupby(["amfi_code", "sector"])["weight_pct"].sum().reset_index())
sector_summary["sector_weight"] = (sector_summary["weight_pct"] / 100)
sector_summary.head()

In [ ]:
sector_hhi = (sector_summary
    .groupby("amfi_code")["sector_weight"]
    .apply(lambda weights:np.sum(weights ** 2))
    .reset_index(name="hhi_score")
)
sector_hhi.head()

In [ ]:
scheme_lookup = (scheme_data[["amfi_code", "scheme_name"]].drop_duplicates())
sector_hhi = (sector_hhi.merge(scheme_lookup,on="amfi_code",how="left"))
sector_hhi.head()

In [ ]:
sector_hhi = (sector_hhi.sort_values(by="hhi_score",ascending=False))
def classify_concentration(hhi):
    if hhi >= 0.25:
        return "Highly Concentrated"
    elif hhi >= 0.15:
        return "Moderately Concentrated"
    else:
        return "Diversified"

sector_hhi["concentration_level"] = (sector_hhi["hhi_score"].apply(classify_concentration))
top_concentrated_funds = (sector_hhi.head(10))
top_concentrated_funds[["scheme_name","hhi_score","concentration_level"]]

In [ ]:
chart_data = sector_hhi.head(10)
plt.figure(figsize=(12, 6))
plt.bar(chart_data["scheme_name"],chart_data["hhi_score"])
plt.title("Top 10 Most Concentrated Funds by Sector HHI")
plt.xlabel("Fund")
plt.ylabel("HHI Score")
plt.xticks(rotation=75,ha="right")
plt.tight_layout()
plt.savefig("../reports/sector_hhi_chart.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
sector_hhi.to_csv("../reports/sector_hhi.csv",index=False)
print("sector_hhi.csv saved successfully.")

# DAY 6 - TASK 7
# Advanced Analytics Summary & Key Insights

# Advanced Analytics Summary

## Insight 1 — Downside Risk Assessment

Historical Value at Risk (VaR) analysis highlighted the funds that are most vulnerable to adverse market movements. Funds with the lowest VaR values exhibited greater downside exposure, indicating a higher probability of experiencing significant short-term losses during volatile market conditions.


## Insight 2 — Tail Risk Evaluation

Conditional Value at Risk (CVaR) provided additional insight into extreme loss scenarios. Certain schemes showed substantially lower CVaR values compared to peers, suggesting that losses can become considerably larger once the VaR threshold is breached. These funds require closer risk monitoring during market stress periods.


## Insight 3 — Investor Cohort Behaviour

Cohort analysis revealed that earlier investor groups contributed the largest share of invested capital. Investors who entered the market in earlier years demonstrated stronger participation levels and generated higher cumulative investment values compared to more recent cohorts.


## Insight 4 — SIP Continuity Trends

SIP continuity analysis indicated that the majority of investors maintained regular contribution patterns. However, a subset of investors exhibited average contribution gaps exceeding 35 days and were classified as At-Risk. These investors may require engagement initiatives to improve long-term retention.


## Insight 5 — Portfolio Concentration Risk

Sector concentration analysis using the Herfindahl-Hirschman Index (HHI) showed meaningful differences in diversification across funds. Funds with elevated HHI scores were more heavily concentrated in a limited number of sectors, increasing sensitivity to sector-specific market movements. Diversified funds generally displayed lower concentration risk and broader sector exposure.